# Golden answers for the Amazon sample

This notebook is the independent answer key for Saabit. It uses plain pandas only and never imports
the app, so if the app and this notebook agree it is not because they share a bug.

Every value below is printed exactly as computed. Nothing is rounded, adjusted or typed in to match
the deck. `eval/golden.yaml` is filled in from these printed values.

## a. Load the file

**What:** read the raw CSV and count its rows.

**Why:** every later number depends on reading the whole file. Some headers carry stray spaces
(`Sales Channel ` ends with one), so headers are stripped before any column is used by name.

In [1]:
from pathlib import Path

import pandas as pd

CSV_PATH = Path("..") / "data" / "sample" / "amazon_sale_report.csv"

df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = df.columns.str.strip()

print("rows:", len(df))
print("columns:", list(df.columns))

rows: 128975
columns: ['index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids', 'B2B', 'fulfilled-by', 'Unnamed: 22']


## b. Parse dates

**What:** parse `Date` with the fixed format month-day-year (`%m-%d-%y`), then print the first and
last date, rows per month, and how many rows fall in March.

**Why:** letting pandas guess the format could read 04-05-22 as 4 May or 5 April. A fixed format
fails loudly instead of guessing. March matters because it is a sliver of a month and would look
like a collapse on a trend chart.

In [11]:
df["order_date"] = pd.to_datetime(df["Date"], format="%m-%d-%y")
df["month"] = df["order_date"].dt.to_period("M")

print("unparsed dates:", df["order_date"].isna().sum())
print("min date:", df["order_date"].min().date())
print("max date:", df["order_date"].max().date())
print()
print("rows per month:")
print(df["month"].value_counts().sort_index().to_string())
print()
print("March rows:", (df["month"] == pd.Period("2022-03", freq="M")).sum())

unparsed dates: 0
min date: 2022-03-31
max date: 2022-06-29

rows per month:
month
2022-03      171
2022-04    49067
2022-05    42040
2022-06    37697
Freq: M

March rows: 171


## c. Orders versus rows

**What:** count unique `Order ID` values and how many rows are extra lines of an order already
counted.

It also counts unique orders in each month.

**Why:** one order can have several item lines. Counting rows as orders inflates every order count.

In [3]:
unique_orders = df["Order ID"].nunique()
lines_per_order = df["Order ID"].value_counts()

print("unique orders:", unique_orders)
print("rows minus unique orders:", len(df) - unique_orders)
print("orders with more than one line:", (lines_per_order > 1).sum())
print("orders with lines in more than one month:", (df.groupby("Order ID")["month"].nunique() > 1).sum())
print()
print("unique orders per month:")
print(df.groupby("month")["Order ID"].nunique().to_string())

unique orders: 120378
rows minus unique orders: 8597
orders with more than one line: 6846
orders with lines in more than one month: 0

unique orders per month:
month
2022-03      158
2022-04    45858
2022-05    39221
2022-06    35141
Freq: M


## d. Cancelled orders: two possible rules

**What:** a line is cancelled when `Status` is exactly `"Cancelled"`. An order can then be called
cancelled in two ways:

- **ANY rule:** the order is cancelled if any of its lines is cancelled.
- **ALL rule:** the order is cancelled only if every one of its lines is cancelled.

Both order-level rates are printed, plus the plain row-level rate for comparison.

**Why:** the two rules differ only on orders with several lines, but that is enough to move the
headline rate. The app must use one rule everywhere (PRD FR-3.3), so this cell shows which one the
deck used.

In [4]:
def pct(part: int, whole: int) -> float:
    return round(100 * part / whole, 4)


df["line_cancelled"] = df["Status"] == "Cancelled"
order_lines = df.groupby("Order ID")["line_cancelled"]
order_any = order_lines.any()
order_all = order_lines.all()
total_orders = len(order_any)

print(f"ANY rule: {order_any.sum()} of {total_orders} orders cancelled = {pct(order_any.sum(), total_orders)}%")
print(f"ALL rule: {order_all.sum()} of {total_orders} orders cancelled = {pct(order_all.sum(), total_orders)}%")
print("orders where the two rules disagree:", (order_any != order_all).sum())
print()
print(f"row-level (not order-level): {df['line_cancelled'].sum()} of {len(df)} rows = {pct(df['line_cancelled'].sum(), len(df))}%")

ANY rule: 17185 of 120378 orders cancelled = 14.2759%
ALL rule: 17185 of 120378 orders cancelled = 14.2759%
orders where the two rules disagree: 0

row-level (not order-level): 18332 of 128975 rows = 14.2136%


## e. Revenue by month

**What:** revenue is the sum of `Amount` over lines whose **order** is not cancelled (PRD metric
definition). It is computed by month, once for each rule. Blank amounts count as zero, and how many
there are is printed.

**Why:** cancelled lines still carry an amount in this file, so including them overstates revenue.
Under the ANY rule, a shipped line inside a cancelled order is also excluded.

In [5]:
df["order_cancelled_any"] = df["Order ID"].map(order_any)
df["order_cancelled_all"] = df["Order ID"].map(order_all)

print("rows with blank Amount:", df["Amount"].isna().sum())
print("blank Amount on non-cancelled lines (ANY rule):", df.loc[~df["order_cancelled_any"], "Amount"].isna().sum())
print()

revenue = pd.DataFrame({
    "ANY rule": df.loc[~df["order_cancelled_any"]].groupby("month")["Amount"].sum(),
    "ALL rule": df.loc[~df["order_cancelled_all"]].groupby("month")["Amount"].sum(),
})
print(revenue.to_string(float_format=lambda v: f"{v:.2f}"))
print()
print("total ANY rule:", round(revenue["ANY rule"].sum(), 2))
print("total ALL rule:", round(revenue["ALL rule"].sum(), 2))

rows with blank Amount: 7795
blank Amount on non-cancelled lines (ANY rule): 229

           ANY rule    ALL rule
month                          
2022-03    94810.00    94810.00
2022-04 26234520.00 26234520.00
2022-05 23953534.00 23953534.00
2022-06 21390530.00 21390530.00

total ANY rule: 71673394.0
total ALL rule: 71673394.0


## f. Value on cancelled lines

**What:** the sum of `Amount` on lines whose status is `Cancelled`, and how many of those lines
have no amount at all.

**Why:** the deck says "at least" a certain value was cancelled. Blank amounts on cancelled lines
are the reason it can only be a lower bound.

In [6]:
cancelled_lines = df.loc[df["line_cancelled"]]

print("Amount on cancelled lines:", round(cancelled_lines["Amount"].sum(), 2))
print("cancelled lines:", len(cancelled_lines))
print("cancelled lines with blank Amount:", cancelled_lines["Amount"].isna().sum())
print("Amount on all lines of ANY-rule cancelled orders:", round(df.loc[df["order_cancelled_any"], "Amount"].sum(), 2))

Amount on cancelled lines: 6919284.3
cancelled lines: 18332
cancelled lines with blank Amount: 7566
Amount on all lines of ANY-rule cancelled orders: 6919284.3


## g. State names

**What:** list every raw `ship-state` spelling, then map each one to a canonical state or union
territory using the dictionary written out below. Anything not in the dictionary is printed and left
unmapped, never guessed. Then Rajasthan's orders and cancellation rate, and the top state by orders.

**Why:** the raw column splits one state across several spellings (RJ, Rajsthan, Rajshthan), so a
naive group-by undercounts it. The dictionary is written by hand here, separately from the app's
own `states.json`.

In [7]:
STATES = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh", "Goa", "Gujarat",
    "Haryana", "Himachal Pradesh", "Jharkhand", "Karnataka", "Kerala", "Madhya Pradesh",
    "Maharashtra", "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Punjab", "Rajasthan",
    "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh", "Uttarakhand", "West Bengal",
]
UNION_TERRITORIES = [
    "Andaman and Nicobar Islands", "Chandigarh", "Dadra and Nagar Haveli and Daman and Diu",
    "Delhi", "Jammu and Kashmir", "Ladakh", "Lakshadweep", "Puducherry",
]
VARIANTS = {
    "rj": "Rajasthan",
    "rajsthan": "Rajasthan",
    "rajshthan": "Rajasthan",
    "orissa": "Odisha",
    "pondicherry": "Puducherry",
    "new delhi": "Delhi",
    "nl": "Nagaland",
    "pb": "Punjab",
    "ar": "Arunachal Pradesh",
    "punjab/mohali/zirakpur": "Punjab",
    "jammu & kashmir": "Jammu and Kashmir",
    "andaman & nicobar": "Andaman and Nicobar Islands",
    "dadra and nagar": "Dadra and Nagar Haveli and Daman and Diu",
}
LOOKUP = {name.lower(): name for name in STATES + UNION_TERRITORIES} | VARIANTS

raw_state = df["ship-state"]
print("distinct raw spellings (blank excluded):", raw_state.nunique())
print("rows with blank state:", raw_state.isna().sum())
print()
print(raw_state.value_counts().to_string())

distinct raw spellings (blank excluded): 69
rows with blank state: 33

ship-state
MAHARASHTRA               22260
KARNATAKA                 17326
TAMIL NADU                11483
TELANGANA                 11330
UTTAR PRADESH             10638
DELHI                      6782
KERALA                     6585
WEST BENGAL                5963
ANDHRA PRADESH             5430
Gujarat                    4489
HARYANA                    4415
RAJASTHAN                  2650
MADHYA PRADESH             2529
ODISHA                     2115
BIHAR                      2086
PUNJAB                     1869
ASSAM                      1663
UTTARAKHAND                1553
JHARKHAND                  1456
GOA                        1102
CHHATTISGARH                909
HIMACHAL PRADESH            788
JAMMU & KASHMIR             702
PUDUCHERRY                  349
CHANDIGARH                  322
MANIPUR                     311
ANDAMAN & NICOBAR           257
MEGHALAYA                   204
SIKKIM                

**Next:** apply the dictionary, show any spelling it did not recognise, then count Rajasthan's
orders, its cancellation rate under both rules, and the state with the most orders. Orders are
counted as distinct `Order ID`s, not rows.

In [8]:
df["state"] = raw_state.str.strip().str.lower().map(LOOKUP)

unmapped = df.loc[df["state"].isna() & raw_state.notna(), "ship-state"].value_counts()
print("unmapped spellings (left as-is):")
print(unmapped.to_string() if len(unmapped) else "none")
print()

states_per_order = df.dropna(subset=["state"]).groupby("Order ID")["state"].nunique()
print("orders with lines in more than one state:", (states_per_order > 1).sum())

orders_by_state = df.dropna(subset=["state"]).groupby("state")["Order ID"].nunique().sort_values(ascending=False)
print("canonical states and UTs present:", len(orders_by_state))
print()

rajasthan_ids = df.loc[df["state"] == "Rajasthan", "Order ID"].unique()
rj_orders = len(rajasthan_ids)
print("Rajasthan orders:", rj_orders)
print(f"Rajasthan cancellation, ANY rule: {order_any.loc[rajasthan_ids].sum()} orders = {pct(order_any.loc[rajasthan_ids].sum(), rj_orders)}%")
print(f"Rajasthan cancellation, ALL rule: {order_all.loc[rajasthan_ids].sum()} orders = {pct(order_all.loc[rajasthan_ids].sum(), rj_orders)}%")
print()
print("top state by orders:", orders_by_state.index[0], orders_by_state.iloc[0])
print()
print("orders by state:")
print(orders_by_state.to_string())

unmapped spellings (left as-is):
ship-state
APO    1

orders with lines in more than one state: 0
canonical states and UTs present: 36

Rajasthan orders: 2512
Rajasthan cancellation, ANY rule: 357 orders = 14.2118%
Rajasthan cancellation, ALL rule: 357 orders = 14.2118%

top state by orders: Maharashtra 20780

orders by state:
state
Maharashtra                                 20780
Karnataka                                   16182
Tamil Nadu                                  10519
Telangana                                   10405
Uttar Pradesh                               10062
Delhi                                        6609
Kerala                                       6105
West Bengal                                  5653
Andhra Pradesh                               4979
Gujarat                                      4169
Haryana                                      4142
Rajasthan                                    2512
Madhya Pradesh                               2375
Odisha         

## h. Cancellation by fulfilment

**What:** cancellation rate and order count for Amazon-fulfilled versus Merchant-fulfilled orders,
for all months, for April-May only, and for June only (the held-out month in the PRD backtest).
Orders are counted once per fulfilment type they have lines in; how many orders mix both types is
printed so that choice is visible.

**Why:** this is the deck's main recommendation. The April-May versus June split is the backtest:
the gap has to hold in a month the rule never saw.

In [9]:
def fulfilment_split(rows: pd.DataFrame) -> pd.DataFrame:
    pairs = rows[["Order ID", "Fulfilment"]].drop_duplicates()
    pairs = pairs.assign(
        cancelled_any=pairs["Order ID"].map(order_any),
        cancelled_all=pairs["Order ID"].map(order_all),
    )
    out = pairs.groupby("Fulfilment").agg(
        orders=("Order ID", "size"),
        cancelled_any=("cancelled_any", "sum"),
        cancelled_all=("cancelled_all", "sum"),
    )
    out["rate_any_%"] = (100 * out["cancelled_any"] / out["orders"]).round(4)
    out["rate_all_%"] = (100 * out["cancelled_all"] / out["orders"]).round(4)
    return out


fulfilment_per_order = df.groupby("Order ID")["Fulfilment"].nunique()
print("orders with both fulfilment types:", (fulfilment_per_order > 1).sum())

PERIODS = {
    "all months": df,
    "Apr-May 2022": df[df["month"].isin([pd.Period("2022-04", freq="M"), pd.Period("2022-05", freq="M")])],
    "Jun 2022": df[df["month"] == pd.Period("2022-06", freq="M")],
}
for name, rows in PERIODS.items():
    print()
    print(f"--- {name}")
    print(fulfilment_split(rows).to_string())

orders with both fulfilment types: 0

--- all months
            orders  cancelled_any  cancelled_all  rate_any_%  rate_all_%
Fulfilment                                                              
Amazon       84002          10803          10803     12.8604     12.8604
Merchant     36376           6382           6382     17.5445     17.5445

--- Apr-May 2022
            orders  cancelled_any  cancelled_all  rate_any_%  rate_all_%
Fulfilment                                                              
Amazon       58170           7483           7483     12.8640     12.8640
Merchant     26909           4727           4727     17.5666     17.5666

--- Jun 2022
            orders  cancelled_any  cancelled_all  rate_any_%  rate_all_%
Fulfilment                                                              
Amazon       25697           3308           3308     12.8731     12.8731
Merchant      9444           1649           1649     17.4608     17.4608


## i. Fulfilment split inside the top 4 categories

**What:** the same fulfilment split, repeated inside each of the four categories with the most
orders, for each period.

**Why:** Merchant might simply sell more of a category that gets cancelled a lot. If the Merchant
gap keeps its sign inside each category, product mix alone does not explain it (PRD D14).

In [10]:
orders_by_category = df.groupby("Category")["Order ID"].nunique().sort_values(ascending=False)
top_categories = list(orders_by_category.index[:4])
print("orders by category:")
print(orders_by_category.to_string())
print()
print("top 4:", top_categories)

for category in top_categories:
    for name, rows in PERIODS.items():
        split = fulfilment_split(rows[rows["Category"] == category])
        gap_any = split.loc["Merchant", "rate_any_%"] - split.loc["Amazon", "rate_any_%"]
        gap_all = split.loc["Merchant", "rate_all_%"] - split.loc["Amazon", "rate_all_%"]
        print()
        print(f"--- {category} | {name} | Merchant minus Amazon: ANY {gap_any:.4f} pts, ALL {gap_all:.4f} pts")
        print(split.to_string())

orders by category:
Category
Set              47845
kurta            46561
Western Dress    14994
Top              10155
Ethnic Dress      1148
Blouse             897
Bottom             410
Saree              144
Dupatta              2

top 4: ['Set', 'kurta', 'Western Dress', 'Top']

--- Set | all months | Merchant minus Amazon: ANY 4.4247 pts, ALL 4.4247 pts
            orders  cancelled_any  cancelled_all  rate_any_%  rate_all_%
Fulfilment                                                              
Amazon       33824           4503           4503     13.3130     13.3130
Merchant     14021           2487           2487     17.7377     17.7377

--- Set | Apr-May 2022 | Merchant minus Amazon: ANY 4.5620 pts, ALL 4.5620 pts
            orders  cancelled_any  cancelled_all  rate_any_%  rate_all_%
Fulfilment                                                              
Amazon       24029           3189           3189     13.2715     13.2715
Merchant     10413           1857           18

## Rule locked for Saabit

Saabit marks an order as cancelled if ANY of its lines has `Status` exactly `Cancelled`; on this file that rule gives 14.2759% (17,185 of 120,378 orders), identical to the ALL rule because no order mixes cancelled and non-cancelled lines.